<a href="https://colab.research.google.com/github/Varmalikith/Supply-chain-management-analysis/blob/main/Supply_chain_management_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')


# Define the folder path where you want to save charts
# Since you are in a cloud workspace, saving to a local 'charts/' folder is easiest
CHART = 'charts/'

# This line checks if the folder exists, and if not, creates it!
os.makedirs(CHART, exist_ok=True)


In [2]:
PALETTE = {
    'primary'   : '#1A6B3C',   # deep green
    'secondary' : '#F5A623',   # amber
    'danger'    : '#D0021B',   # red
    'accent'    : '#2C7BB6',   # blue
    'light'     : '#F7F9FC',
    'dark'      : '#1C2B36',
    'grid'      : '#E5EAF0',
}

plt.rcParams.update({
    'figure.facecolor' : PALETTE['light'],
    'axes.facecolor'   : PALETTE['light'],
    'axes.edgecolor'   : '#C8D0DA',
    'axes.labelcolor'  : PALETTE['dark'],
    'axes.titlecolor'  : PALETTE['dark'],
    'axes.titlesize'   : 14,
    'axes.labelsize'   : 11,
    'axes.titleweight' : 'bold',
    'xtick.color'      : '#4A5568',
    'ytick.color'      : '#4A5568',
    'font.family'      : 'DejaVu Sans',
    'axes.grid'        : True,
    'grid.color'       : PALETTE['grid'],
    'grid.linestyle'   : '--',
    'grid.linewidth'   : 0.6,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'legend.frameon'   : False,
})

CHART = '/home/claude/supply_chain_project/charts/'


In [9]:
print("=" * 60)
print("STEP 1: DATA LOADING & FIRST LOOK")
print("=" * 60)

df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding='unicode_escape')
print(f"\nDataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nFirst 5 rows (key columns):")
print(df[['Order Id','Category Name','Delivery Status','Sales','Shipping Mode','Late_delivery_risk']].head())


STEP 1: DATA LOADING & FIRST LOOK

Dataset Shape: 33,762 rows × 53 columns

First 5 rows (key columns):
   Order Id   Category Name   Delivery Status   Sales   Shipping Mode  \
0   77202.0  Sporting Goods  Advance shipping  327.75  Standard Class   
1   75939.0  Sporting Goods     Late delivery  327.75  Standard Class   
2   75938.0  Sporting Goods  Shipping on time  327.75  Standard Class   
3   75937.0  Sporting Goods  Advance shipping  327.75  Standard Class   
4   75936.0  Sporting Goods  Advance shipping  327.75  Standard Class   

   Late_delivery_risk  
0                   0  
1                   1  
2                   0  
3                   0  
4                   0  


In [10]:
print("\n" + "=" * 60)
print("STEP 2: EXPLORATORY DATA ANALYSIS")
print("=" * 60)
print("\nBasic Statistics:")
print(df[['Sales','Benefit per order','Order Item Quantity','Days for shipping (real)','Order Item Profit Ratio']].describe().round(2))
print("\nMissing Values:", df.isnull().sum()[df.isnull().sum()>0].to_dict())
print("\nDelivery Status Distribution:")
print(df['Delivery Status'].value_counts())


STEP 2: EXPLORATORY DATA ANALYSIS

Basic Statistics:
          Sales  Benefit per order  Order Item Quantity  \
count  33761.00           33762.00             33761.00   
mean     196.54              21.52                 2.44   
std      137.63             104.77                 1.56   
min        9.99           -3366.00                 1.00   
25%      110.00               6.25                 1.00   
50%      159.96              29.13                 2.00   
75%      299.95              62.39                 4.00   
max     1999.99             684.00                 5.00   

       Days for shipping (real)  Order Item Profit Ratio  
count                  33762.00                 33761.00  
mean                       3.46                     0.12  
std                        1.59                     0.46  
min                        0.00                    -2.75  
25%                        2.00                     0.08  
50%                        3.00                     0.27  
7

In [12]:
# Print columns containing 'date' or 'shipping' to find the exact match
print([col for col in df.columns if 'date' in col.lower() or 'shipping' in col.lower()])

['Days for shipping (real)', 'order date (DateOrders)', 'shipping date (DateOrders)', 'Shipping Mode']


In [14]:
print("\n" + "=" * 60)
print("STEP 3: DATA CLEANING & FEATURE ENGINEERING")
print("=" * 60)

df['order date (DateOrders)']    = pd.to_datetime(df['order date (DateOrders)'],    errors='coerce')
# FIXED: Changed capital 'S' to lowercase 's' to match your dataset columns
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'], errors='coerce')

df['Shipping Delay']    = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
df['Is_Late']           = (df['Shipping Delay'] > 0).astype(int)
df['Year']              = df['order date (DateOrders)'].dt.year
df['Month']             = df['order date (DateOrders)'].dt.month
df['Quarter']           = df['order date (DateOrders)'].dt.quarter
df['Month_Name']        = df['order date (DateOrders)'].dt.strftime('%b')
df['Profit_Flag']       = (df['Benefit per order'] > 0).astype(int)
df['Revenue_per_unit']  = df['Sales'] / df['Order Item Quantity'].replace(0,1)

print(f"New features added: Shipping Delay, Is_Late, Year, Month, Quarter, Month_Name, Profit_Flag, Revenue_per_unit")
print(f"Late delivery rate: {df['Is_Late'].mean()*100:.1f}%")
print(f"Profitable orders: {df['Profit_Flag'].mean()*100:.1f}%")


STEP 3: DATA CLEANING & FEATURE ENGINEERING
New features added: Shipping Delay, Is_Late, Year, Month, Quarter, Month_Name, Profit_Flag, Revenue_per_unit
Late delivery rate: 58.7%
Profitable orders: 80.8%


In [18]:
# ── CHART 1: Delivery Status Distribution ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])
fig.suptitle('Delivery Status Overview', fontsize=16, fontweight='bold', color=PALETTE['dark'], y=1.02)

del_counts = df['Delivery Status'].value_counts()
colors_del = [PALETTE['danger'] if 'Late' in x else PALETTE['primary'] if 'Advance' in x
              else PALETTE['secondary'] if 'on time' in x else '#A0AEC0' for x in del_counts.index]
bars = axes[0].bar(del_counts.index, del_counts.values, color=colors_del, edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_title('Delivery Status Count', pad=10)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, del_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'{val:,}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold', color=PALETTE['dark'])

wedge_colors = [PALETTE['danger'], PALETTE['primary'], PALETTE['secondary'], '#A0AEC0']
wedges, texts, autotexts = axes[1].pie(del_counts.values, labels=None, autopct='%1.1f%%',
                                        colors=wedge_colors[:len(del_counts)], startangle=90,
                                        pctdistance=0.75, wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(9); at.set_fontweight('bold')
axes[1].legend(del_counts.index, loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=9)
axes[1].set_title('Delivery Status Share', pad=10)

plt.tight_layout()
plt.savefig(f'{CHART}01_delivery_status.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("\n[✓] Chart 1 saved: Delivery Status")


[✓] Chart 1 saved: Delivery Status


In [19]:
# ── CHART 2: Sales by Category ───────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(PALETTE['light'])

cat_sales = df.groupby('Category Name')['Sales'].sum().sort_values(ascending=True)
cmap_vals = np.linspace(0.3, 1.0, len(cat_sales))
bar_colors = [plt.cm.YlGn(v) for v in cmap_vals]
bars = ax.barh(cat_sales.index, cat_sales.values, color=bar_colors, edgecolor='white', linewidth=1)
ax.set_title('Total Sales by Product Category', pad=12)
ax.set_xlabel('Total Sales (USD)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
for bar, val in zip(bars, cat_sales.values):
    ax.text(val + cat_sales.max()*0.01, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=8.5, color=PALETTE['dark'])
ax.set_xlim(0, cat_sales.max() * 1.15)
plt.tight_layout()
plt.savefig(f'{CHART}02_sales_by_category.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 2 saved: Sales by Category")

[✓] Chart 2 saved: Sales by Category


In [20]:
# ── CHART 3: Shipping Mode vs Late Delivery ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])
fig.suptitle('Shipping Mode Analysis', fontsize=16, fontweight='bold', color=PALETTE['dark'])

late_by_mode = df.groupby('Shipping Mode')['Is_Late'].mean().mul(100).sort_values(ascending=False)
colors_mode = [PALETTE['danger'] if v > 50 else PALETTE['secondary'] if v > 30 else PALETTE['primary']
               for v in late_by_mode.values]
bars = axes[0].bar(late_by_mode.index, late_by_mode.values, color=colors_mode, edgecolor='white', linewidth=1.5, width=0.55)
axes[0].set_title('Late Delivery Rate by Shipping Mode (%)')
axes[0].set_ylabel('Late Delivery Rate (%)')
axes[0].set_ylim(0, late_by_mode.max() * 1.2)
for bar, val in zip(bars, late_by_mode.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.1f}%',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

vol_mode = df['Shipping Mode'].value_counts()
axes[1].bar(vol_mode.index, vol_mode.values, color=PALETTE['accent'], edgecolor='white', linewidth=1.5, width=0.55, alpha=0.85)
axes[1].set_title('Order Volume by Shipping Mode')
axes[1].set_ylabel('Number of Orders')
for bar, val in zip(axes[1].patches, vol_mode.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, f'{val:,}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(f'{CHART}03_shipping_mode_analysis.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 3 saved: Shipping Mode Analysis")

[✓] Chart 3 saved: Shipping Mode Analysis


In [21]:
# ── CHART 4: Market Performance ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])
fig.suptitle('Market Performance Analysis', fontsize=16, fontweight='bold', color=PALETTE['dark'])

mkt_sales  = df.groupby('Market')['Sales'].sum().sort_values(ascending=False)
mkt_profit = df.groupby('Market')['Benefit per order'].sum().sort_values(ascending=False)

bar_colors_mkt = [PALETTE['primary'], PALETTE['accent'], PALETTE['secondary'], '#8B5CF6', '#EC4899']
axes[0].bar(mkt_sales.index, mkt_sales.values, color=bar_colors_mkt[:len(mkt_sales)], edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_title('Total Sales by Market')
axes[0].set_ylabel('Total Sales (USD)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000,
                 f'${bar.get_height()/1e6:.2f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

colors_profit = [PALETTE['primary'] if v > 0 else PALETTE['danger'] for v in mkt_profit.values]
axes[1].bar(mkt_profit.index, mkt_profit.values, color=colors_profit, edgecolor='white', linewidth=1.5, width=0.6)
axes[1].set_title('Total Profit by Market')
axes[1].set_ylabel('Total Profit (USD)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(f'{CHART}04_market_performance.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 4 saved: Market Performance")

[✓] Chart 4 saved: Market Performance


In [22]:
# ── CHART 5: Monthly Sales Time Series ──────────────────
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])

monthly = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
monthly['Date'] = pd.to_datetime(monthly[['Year','Month']].assign(DAY=1))
monthly = monthly.sort_values('Date')

ax.fill_between(monthly['Date'], monthly['Sales'], alpha=0.15, color=PALETTE['primary'])
ax.plot(monthly['Date'], monthly['Sales'], color=PALETTE['primary'], linewidth=2.5, marker='o', markersize=4)
ax.set_title('Monthly Sales Trend (Time Series)', pad=12)
ax.set_xlabel('Date')
ax.set_ylabel('Monthly Sales (USD)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))

# Add 3-month rolling average
monthly['Rolling_Avg'] = monthly['Sales'].rolling(3, min_periods=1).mean()
ax.plot(monthly['Date'], monthly['Rolling_Avg'], color=PALETTE['secondary'], linewidth=2,
        linestyle='--', label='3-Month Rolling Avg', alpha=0.9)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{CHART}05_time_series.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 5 saved: Time Series")

[✓] Chart 5 saved: Time Series


In [23]:
# ── CHART 6: Bottleneck Detection ───────────────────────
print("\n" + "=" * 60)
print("STEP 4: BOTTLENECK DETECTION")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])
fig.suptitle('Bottleneck Detection', fontsize=16, fontweight='bold', color=PALETTE['dark'])

delay_region = df.groupby('Order Region')['Shipping Delay'].mean().sort_values(ascending=False).head(10)
colors_delay = [PALETTE['danger'] if v > 1 else PALETTE['secondary'] if v > 0 else PALETTE['primary']
                for v in delay_region.values]
axes[0].barh(delay_region.index, delay_region.values, color=colors_delay, edgecolor='white', linewidth=1)
axes[0].set_title('Avg Shipping Delay by Region (Top 10)', pad=8)
axes[0].set_xlabel('Average Delay (Days)')
axes[0].axvline(0, color='#718096', linewidth=1, linestyle='-')
for bar, val in zip(axes[0].patches, delay_region.values):
    axes[0].text(val + 0.02 if val >= 0 else val - 0.02, bar.get_y()+bar.get_height()/2,
                 f'{val:+.2f}d', va='center', fontsize=8)

delay_cat = df.groupby('Category Name')['Is_Late'].mean().mul(100).sort_values(ascending=False)
colors_cat = [PALETTE['danger'] if v > 55 else PALETTE['secondary'] if v > 45 else PALETTE['primary']
              for v in delay_cat.values]
bars = axes[1].bar(range(len(delay_cat)), delay_cat.values, color=colors_cat, edgecolor='white', linewidth=1, width=0.6)
axes[1].set_xticks(range(len(delay_cat)))
axes[1].set_xticklabels(delay_cat.index, rotation=35, ha='right', fontsize=8)
axes[1].set_title('Late Delivery Rate by Category (%)', pad=8)
axes[1].set_ylabel('Late Rate (%)')
axes[1].axhline(50, color=PALETTE['danger'], linewidth=1.5, linestyle='--', label='50% threshold')
axes[1].legend(fontsize=9)

print(f"Highest delay region: {delay_region.index[0]} ({delay_region.values[0]:+.2f} days)")
print(f"Highest late rate category: {delay_cat.index[0]} ({delay_cat.values[0]:.1f}%)")

plt.tight_layout()
plt.savefig(f'{CHART}06_bottleneck_detection.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 6 saved: Bottleneck Detection")


STEP 4: BOTTLENECK DETECTION
Highest delay region: Central Asia (+0.79 days)
Highest late rate category: Basketball (72.7%)
[✓] Chart 6 saved: Bottleneck Detection


In [24]:
# ── CHART 7: Root Cause Analysis ────────────────────────
print("\n" + "=" * 60)
print("STEP 5: ROOT CAUSE ANALYSIS")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])
fig.suptitle('Root Cause Analysis — Late Deliveries', fontsize=16, fontweight='bold', color=PALETTE['dark'])

late_by_seg = df.groupby('Customer Segment')['Is_Late'].mean().mul(100)
axes[0].bar(late_by_seg.index, late_by_seg.values,
            color=[PALETTE['primary'], PALETTE['accent'], PALETTE['secondary']],
            edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('Late Delivery Rate by Customer Segment')
axes[0].set_ylabel('Late Rate (%)')
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

corr_cols = ['Days for shipping (real)', 'Days for shipment (scheduled)',
             'Shipping Delay', 'Order Item Quantity', 'Sales', 'Order Item Profit Ratio']
corr = df[corr_cols + ['Is_Late']].corr()['Is_Late'].drop('Is_Late').sort_values()
colors_corr = [PALETTE['danger'] if v > 0 else PALETTE['primary'] for v in corr.values]
axes[1].barh(corr.index, corr.values, color=colors_corr, edgecolor='white', linewidth=1)
axes[1].axvline(0, color='#718096', linewidth=1)
axes[1].set_title('Correlation with Late Delivery Risk')
axes[1].set_xlabel('Correlation Coefficient')
for bar, val in zip(axes[1].patches, corr.values):
    axes[1].text(val + 0.005 if val >= 0 else val - 0.005,
                 bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

print(f"\nTop late delivery driver: {corr.abs().idxmax()} (r={corr.abs().max():.3f})")

plt.tight_layout()
plt.savefig(f'{CHART}07_root_cause_analysis.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 7 saved: Root Cause Analysis")



STEP 5: ROOT CAUSE ANALYSIS

Top late delivery driver: Shipping Delay (r=0.818)
[✓] Chart 7 saved: Root Cause Analysis


In [25]:
# ── CHART 8: Profit Analysis ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])
fig.suptitle('Profitability Analysis', fontsize=16, fontweight='bold', color=PALETTE['dark'])

axes[0].hist(df['Order Item Profit Ratio'], bins=30, color=PALETTE['primary'], edgecolor='white', linewidth=0.5, alpha=0.85)
axes[0].axvline(df['Order Item Profit Ratio'].mean(), color=PALETTE['danger'], linewidth=2, linestyle='--',
                label=f"Mean: {df['Order Item Profit Ratio'].mean():.2f}")
axes[0].axvline(0, color='#718096', linewidth=1.5, linestyle='-', label='Break-even')
axes[0].set_title('Distribution of Profit Ratio')
axes[0].set_xlabel('Profit Ratio')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=9)

dept_profit = df.groupby('Department Name')[['Sales','Benefit per order']].sum()
dept_profit['Margin%'] = (dept_profit['Benefit per order'] / dept_profit['Sales'] * 100).round(1)
dept_profit = dept_profit.sort_values('Sales', ascending=True)
bars1 = axes[1].barh(dept_profit.index, dept_profit['Sales'], alpha=0.7, color=PALETTE['accent'], label='Sales', height=0.4)
ax2 = axes[1].twiny()
bars2 = ax2.barh([i+0.4 for i in range(len(dept_profit))], dept_profit['Benefit per order'],
                  alpha=0.85, color=PALETTE['secondary'], label='Profit', height=0.4)
axes[1].set_title('Sales vs Profit by Department')
axes[1].set_xlabel('Sales (USD)', color=PALETTE['accent'])
ax2.set_xlabel('Profit (USD)', color=PALETTE['secondary'])
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].legend(loc='lower right', fontsize=9)
ax2.legend(loc='center right', fontsize=9)

plt.tight_layout()
plt.savefig(f'{CHART}08_profit_analysis.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 8 saved: Profit Analysis")

[✓] Chart 8 saved: Profit Analysis


In [26]:
# ── CHART 9: Heatmap — Late Delivery by Month & Mode ────
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor(PALETTE['light'])

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['Month_Num'] = df['Month']
heatmap_data = df.pivot_table(values='Is_Late', index='Shipping Mode', columns='Month_Num', aggfunc='mean') * 100
heatmap_data.columns = [month_names[c] for c in heatmap_data.columns]

sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn_r', linewidths=0.5,
            linecolor='white', ax=ax, cbar_kws={'label': 'Late Rate (%)', 'shrink': 0.8},
            vmin=0, vmax=100, annot_kws={'size': 9, 'weight': 'bold'})
ax.set_title('Late Delivery Rate (%) by Shipping Mode & Month', pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('Shipping Mode')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f'{CHART}09_heatmap_late_delivery.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 9 saved: Late Delivery Heatmap")

[✓] Chart 9 saved: Late Delivery Heatmap


In [27]:
# ── MACHINE LEARNING: PREDICT LATE DELIVERY RISK ────────
print("\n" + "=" * 60)
print("STEP 6: MACHINE LEARNING — LATE DELIVERY RISK PREDICTION")
print("=" * 60)

features = ['Days for shipment (scheduled)', 'Order Item Quantity',
            'Sales', 'Order Item Discount Rate', 'Order Item Profit Ratio',
            'Product Price', 'Shipping Mode', 'Market', 'Customer Segment']

df_ml = df[features + ['Late_delivery_risk']].dropna()
le_dict = {}
for col in ['Shipping Mode', 'Market', 'Customer Segment']:
    le = LabelEncoder()
    df_ml = df_ml.copy()
    df_ml[col] = le.fit_transform(df_ml[col])
    le_dict[col] = le

X = df_ml[features]
y = df_ml['Late_delivery_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {acc*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['On Time', 'Late']))



STEP 6: MACHINE LEARNING — LATE DELIVERY RISK PREDICTION

Model Accuracy: 71.64%

Classification Report:
              precision    recall  f1-score   support

     On Time       0.62      0.87      0.72      2887
        Late       0.86      0.60      0.71      3866

    accuracy                           0.72      6753
   macro avg       0.74      0.74      0.72      6753
weighted avg       0.76      0.72      0.72      6753



In [28]:
# ── CHART 10: Feature Importance ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PALETTE['light'])
fig.suptitle('Machine Learning Model Results', fontsize=16, fontweight='bold', color=PALETTE['dark'])

importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
cmap_fi = np.linspace(0.3, 1.0, len(importances))
fi_colors = [plt.cm.Greens(v) for v in cmap_fi]
axes[0].barh(importances.index, importances.values, color=fi_colors, edgecolor='white', linewidth=1)
axes[0].set_title('Feature Importance — Late Delivery Risk')
axes[0].set_xlabel('Importance Score')
for bar, val in zip(axes[0].patches, importances.values):
    axes[0].text(val + 0.001, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8.5)

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], linewidths=1, linecolor='white',
            xticklabels=['On Time', 'Late'], yticklabels=['On Time', 'Late'],
            annot_kws={'size': 12, 'weight': 'bold'})
axes[1].set_title(f'Confusion Matrix (Accuracy: {acc*100:.1f}%)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{CHART}10_ml_results.png', dpi=150, bbox_inches='tight', facecolor=PALETTE['light'])
plt.close()
print("[✓] Chart 10 saved: ML Results")

[✓] Chart 10 saved: ML Results


In [30]:
# ── KEY METRICS SUMMARY ──────────────────────────────────
print("\n" + "=" * 60)
print("KEY BUSINESS METRICS")
print("=" * 60)

total_sales   = df['Sales'].sum()
total_profit  = df['Benefit per order'].sum()
profit_margin = total_profit / total_sales * 100
late_rate     = df['Is_Late'].mean() * 100
avg_delay     = df[df['Is_Late']==1]['Shipping Delay'].mean()
top_cat       = df.groupby('Category Name')['Sales'].sum().idxmax()
top_mkt       = df.groupby('Market')['Sales'].sum().idxmax()

print(f"  Total Sales:       ${total_sales:>12,.0f}")
print(f"  Total Profit:      ${total_profit:>12,.0f}")
print(f"  Profit Margin:     {profit_margin:>11.1f}%")
print(f"  Late Delivery Rate:{late_rate:>11.1f}%")
print(f"  Avg Delay (days):  {avg_delay:>11.2f}")
print(f"  Top Category:      {top_cat}")
print(f"  Top Market:        {top_mkt}")
print(f"  ML Accuracy:       {acc*100:>11.1f}%")

metrics = {
    'total_sales': total_sales, 'total_profit': total_profit,
    'profit_margin': profit_margin, 'late_rate': late_rate,
    'avg_delay': avg_delay, 'top_cat': top_cat, 'top_mkt': top_mkt, 'ml_acc': acc * 100
}

print("\n[✓] All charts generated. Proceeding to report generation...")

# save metrics for report
import json
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)


KEY BUSINESS METRICS
  Total Sales:       $   6,635,441
  Total Profit:      $     726,447
  Profit Margin:            10.9%
  Late Delivery Rate:       58.7%
  Avg Delay (days):         1.54
  Top Category:      Cleats
  Top Market:        Europe
  ML Accuracy:              71.6%

[✓] All charts generated. Proceeding to report generation...
